# AIQ Pipeline Testing

**Flow:**
1. Load processed results from pipeline_detection
2. Review & fix remaining issues
3. Load test set (questions + ground truth)
4. Run evaluation (BM25 retrieve)
5. Results summary
6. Save results

---
## Cell 1: Configuration

In [4]:
import sys, os, json, csv
if os.path.basename(os.getcwd()) == 'examples':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

RESULTS_FILE = '../results/neuroloft_kb_results.json'
if not os.path.exists(RESULTS_FILE):
    RESULTS_FILE = 'results/neuroloft_kb_results.json'

TEST_SET_FILE = 'test_set.csv'
for p in [TEST_SET_FILE, '../examples/test_set.csv', os.path.join(os.path.dirname(os.getcwd()), 'examples', 'test_set.csv')]:
    if os.path.exists(p):
        TEST_SET_FILE = p
        break

MAX_ROWS = 10

def show(items, fmt, label='items'):
    total = len(items)
    for item in items[:MAX_ROWS]:
        print(fmt(item))
    if total > MAX_ROWS:
        print(f'  ... and {total - MAX_ROWS} more {label}')

print(f'Results: {RESULTS_FILE} (exists: {os.path.exists(RESULTS_FILE)})')
print(f'Test set: {TEST_SET_FILE} (exists: {os.path.exists(TEST_SET_FILE)})')

Results: ../results/neuroloft_kb_results.json (exists: True)
Test set: test_set.csv (exists: True)


---
## Cell 2: Load Processed Results

In [7]:
with open(RESULTS_FILE, encoding='utf-8') as f:
    results = json.load(f)

print(f'Input: {results["input_file"]} | {results["timestamp"]}')
print(f'Chunks: {results["stats"]["final_chunks"]} | Words: {results["stats"]["final_words"]}')
print(f'\nModule summary:')
print(f'  {"Module":<30s} {"Found":>6s} {"Resolved":>9s} {"Remaining":>10s}')
print(f'  {"─" * 55}')
for name, m in results['modules'].items():
    print(f'  {name:<30s} {m["detected"]:>6d} {m["resolved"]:>9d} {m["remaining"]:>10d}')

from aiq.core.types import Chunk
chunks = [Chunk(chunk_id=c['chunk_id'], heading=c.get('heading',''), content=c['content'], words=c.get('words', len(c['content'].split())), source_type='text') for c in results['chunks']]
print(f'\nLoaded {len(chunks)} chunks')

Input: neuroloft_kb.html | 2026-05-05T22:13:31.847696
Chunks: 24 | Words: 2110

Module summary:
  Module                          Found  Resolved  Remaining
  ───────────────────────────────────────────────────────
  Domain Intelligence               171       171          0
  Content Normalization              15        15          0
  Structure & Headings                1         1          0
  Metadata Enrichment                 7         7          0
  Semantic Clarity                   28        19          9
  Content Governance                 49         8          0
  Consistency                         2         0          2

Loaded 24 chunks


---
## Cell 3: Review Remaining Issues

In [10]:
remaining = {name: m for name, m in results['modules'].items() if m['remaining'] > 0}
if remaining:
    print('Remaining issues:')
    for name, m in remaining.items():
        print(f'  {name}: {m["remaining"]} unresolved')
else:
    print('No remaining issues.')

Remaining issues:
  Semantic Clarity: 9 unresolved
  Consistency: 2 unresolved


### ✏️ Fix Remaining Issues

In [13]:
# === RESOLVE REMAINING ISSUES ===
# For each issue: provide a fix or mark as 'skip' (acceptable as-is)

fixes = {
    # Clarity: vague "The team" / "the team"
    'c3': {'find': 'The team', 'replace': 'The billing team'},
    'c7': {'find': 'the team', 'replace': 'the support team'},
    # Clarity: possessive "their" — acceptable in context
    'c4': 'skip',
    'c6': 'skip',
    'c11': 'skip',
    'c15': 'skip',
    'c21': 'skip',
    # Clarity: "These" — acceptable
    'c20': 'skip',
    # Clarity: complex sentence — acceptable (process description)
    'c26': 'skip',
    # Consistency: process conflict — acceptable (different contexts)
    'c10_con': 'skip',
    # Consistency: numeric — acceptable (historical pricing)
    'c20_con': 'skip',
}

# Apply fixes
applied = 0
skipped = 0
for chunk_id, action in fixes.items():
    if action == 'skip':
        skipped += 1
        continue
    for c in chunks:
        if c.chunk_id == chunk_id:
            c.content = c.content.replace(action['find'], action['replace'])
            c.words = len(c.content.split())
            applied += 1

# Verify
remaining_check = sum(m['remaining'] for m in results['modules'].values())
resolved_by_fixes = applied + skipped

if resolved_by_fixes >= remaining_check:
    print(f'All remaining issues addressed: {applied} fixed, {skipped} skipped (acceptable)')
else:
    print(f'WARNING: {remaining_check - resolved_by_fixes} issues still unresolved.')
    print(f'  Applied: {applied}, Skipped: {skipped}, Total remaining: {remaining_check}')

print(f'\n{len(chunks)} chunks ready for testing')

All remaining issues addressed: 2 fixed, 9 skipped (acceptable)

24 chunks ready for testing


---
## Cell 4: Load Test Set

In [16]:
test_pairs = []
if os.path.exists(TEST_SET_FILE):
    with open(TEST_SET_FILE, encoding='utf-8') as f:
        for row in csv.DictReader(f):
            test_pairs.append(row)
    print(f'Loaded: {TEST_SET_FILE} ({len(test_pairs)} pairs)')
    from collections import Counter
    cats = Counter(t['category'] for t in test_pairs)
    for cat, count in cats.most_common():
        print(f'  {cat}: {count}')
    print()
    show(test_pairs, lambda t: f'  [{t["id"]}] {t["category"]}: {t["question"][:70]}', 'pairs')
else:
    print(f'Test set not found: {TEST_SET_FILE}')

Loaded: test_set.csv (45 pairs)
  topic: 18
  governance: 13
  clarity: 9
  consistency: 5

  [T01] topic: What payment methods does Neuroloft support?
  [T02] topic: What are the processing times and fees for credit card payments?
  [T03] topic: How long does a wire transfer take and what is the fee?
  [T04] topic: What features are included in the Professional plan?
  [T05] topic: What does error code ERR-003 mean?
  [T06] topic: What are the steps in the payment processing flow?
  [T07] topic: Can you explain the refund workflow?
  [T08] topic: What is the dispute resolution process?
  [T09] topic: How do I set up a new account?
  [T10] topic: What is the free trial period?
  ... and 35 more pairs


---
## Cell 5: Run Evaluation

In [19]:
from aiq.a42.retrieval import _bm25_score, _tokenize

def bm25_retrieve(question, pool, top_k=3):
    q_tokens = _tokenize(question)
    all_tokens = [_tokenize(c.content) for c in pool]
    avg_dl = sum(len(t) for t in all_tokens) / max(len(all_tokens), 1)
    scored = [(pool[i], _bm25_score(q_tokens, all_tokens[i], avg_dl)) for i in range(len(pool))]
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

eval_results = []
if test_pairs:
    for t in test_pairs:
        retrieved = bm25_retrieve(t['question'], chunks)
        top = retrieved[0][0] if retrieved else None
        eval_results.append({
            'id': t['id'],
            'category': t['category'],
            'question': t['question'],
            'ground_truth': t['ground_truth'],
            'expected_behavior': t['expected_behavior'],
            'retrieved_chunk': top.chunk_id if top else 'none',
            'retrieved_content': top.content[:200] if top else '',
        })
    print(f'Evaluation complete: {len(eval_results)} pairs')
else:
    print('No test set loaded.')

Evaluation complete: 45 pairs


---
## Cell 6: Results Summary

In [22]:
if eval_results:
    from collections import Counter
    
    print(f'Evaluation: {len(eval_results)} test pairs against {len(chunks)} chunks')
    print(f'\n{"─" * 50}')
    print(f'  {"Category":<15s} {"Count":>6s} {"Retrieved":>10s}')
    print(f'  {"─" * 35}')
    
    for cat in ['topic', 'governance', 'clarity', 'consistency']:
        cat_results = [r for r in eval_results if r['category'] == cat]
        if not cat_results: continue
        ok = sum(1 for r in cat_results if r['retrieved_chunk'] != 'none')
        print(f'  {cat:<15s} {len(cat_results):>6d} {ok:>10d}')
    
    total = len(eval_results)
    total_ok = sum(1 for r in eval_results if r['retrieved_chunk'] != 'none')
    print(f'  {"─" * 35}')
    print(f'  {"TOTAL":<15s} {total:>6d} {total_ok:>10d}')
    
    print(f'\nDetailed results:')
    show(eval_results, lambda r: f'  [{r["id"]:>4s}] {r["category"]:>12s} | chunk={r["retrieved_chunk"]:>4s} | Q: {r["question"][:40]}... | GT: {r["ground_truth"][:35]}...', 'results')
else:
    print('No results to display.')

Evaluation: 45 test pairs against 24 chunks

──────────────────────────────────────────────────
  Category         Count  Retrieved
  ───────────────────────────────────
  topic               18         18
  governance          13         13
  clarity              9          9
  consistency          5          5
  ───────────────────────────────────
  TOTAL               45         45

Detailed results:
  [ T01]        topic | chunk=  c4 | Q: What payment methods does Neuroloft supp... | GT: Credit Card (Visa, Mastercard, Amex...
  [ T02]        topic | chunk=  c4 | Q: What are the processing times and fees f... | GT: Credit Card processing is instant w...
  [ T03]        topic | chunk=  c6 | Q: How long does a wire transfer take and w... | GT: 3-5 business days with a $25 flat f...
  [ T04]        topic | chunk= c18 | Q: What features are included in the Profes... | GT: includes up to 25 users, 100 GB sto...
  [ T05]        topic | chunk= c12 | Q: What does error code ERR-003 mean?...

---
## Cell 7: Save Results

In [24]:
if eval_results:
    results_dir = os.path.join(os.path.dirname(os.getcwd()), 'results') if os.path.basename(os.getcwd()) == 'examples' else os.path.join(os.getcwd(), 'results')
    os.makedirs(results_dir, exist_ok=True)
    doc_name = os.path.splitext(os.path.basename(results['input_file']))[0]
    out_path = os.path.join(results_dir, f'{doc_name}_eval.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump({'source': RESULTS_FILE, 'test_set': TEST_SET_FILE, 'total': len(eval_results), 'results': eval_results}, f, indent=2, ensure_ascii=False)
    print(f'Saved: {out_path}')
else:
    print('No results to save.')

Saved: C:\Users\samir\OneDrive\Desktop\KG RAG\opensource\aiq\results\neuroloft_kb_eval.json
